# QLoRA Fine-Tuning: Qwen2.5-Coder-7B-Instruct for SVA Generation

Fine-tunes Qwen2.5-Coder-7B-Instruct using QLoRA to generate SystemVerilog Assertions (SVA) from RTL.

**Ablation Configs** — change `DATASET_CONFIG` below:
- `all` — unfiltered (includes non-compiling)
- `syntax_pass` — compiles in Jasper
- `verified` — compiles AND passes verification

**Requirements**: Colab Pro with A100 GPU.

## 0. Install Dependencies

In [ ]:
!pip install -q \
    torch \
    transformers>=4.44.0 \
    datasets \
    accelerate \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    wandb \
    huggingface_hub

## 1. Configuration

**Change `DATASET_CONFIG` for each ablation run.**

In [ ]:
import os

# ============================================================
# CHANGE THESE BETWEEN ABLATION RUNS
# ============================================================
DATASET_CONFIG = "all"  # Options: "all", "syntax_pass", "verified"
RUN_NAME = f"sva-qlora-{DATASET_CONFIG}"

# ============================================================
# Dataset
# ============================================================
DATASET_REPO = "../veri2" 

# ============================================================
# Model
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ============================================================
# QLoRA Hyperparameters
# ============================================================
LORA_RANK = 64
LORA_ALPHA = 128          # 2x rank is standard
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ============================================================
# Training Hyperparameters
# ============================================================
NUM_EPOCHS = 3
BATCH_SIZE = 2              # per-device
GRADIENT_ACCUMULATION = 8   # effective batch = 2 * 8 = 16
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.05
MAX_SEQ_LENGTH = 8192
WEIGHT_DECAY = 0.01

# ============================================================
# Output
# ============================================================          # points to the local veri2 folder
OUTPUT_DIR = os.path.join("../veri2", "sva_qlora", RUN_NAME)
HF_ADAPTER_REPO = f"sva-qlora-{DATASET_CONFIG}"  # generic name (no username)

## 2. Login & Setup

In [ ]:
# HuggingFace interactive login removed for anonymized local runs.
# If you need to authenticate, run `huggingface-cli login` locally or set HF_TOKEN in the environment.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Optional: Weights & Biases logging (anonymized)
import wandb
# wandb.login()  # login removed for double-blind runs
wandb.init(project="sva-qlora-ablation", name=RUN_NAME)  # entity omitted

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Load & Format Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_REPO, DATASET_CONFIG)
print(f"Config: {DATASET_CONFIG}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

In [ ]:
# ============================================================
# System prompt — same one used for ChatGPT generation
# ============================================================
SYSTEM_PROMPT = """\
You are an expert SystemVerilog verification engineer writing SVA for Jasper \
formal verification.
OUTPUT REQUIREMENTS:
1. Output a single, complete .sv file that compiles in Jasper without modification.
2. The file must be a module that takes the DUT's ports as inputs and contains \
   SVA properties bound to those signals. You can use internal signals \
   if they are present in the RTL, but do NOT invent new ones.
3. Every property must use a clocked event (@(posedge clk) or the appropriate \
   clock from the RTL). NEVER use combinational or level-sensitive events in \
   property statements — Jasper rejects these. For modules with combinational \
   logic, still clock your assertions to the appropriate clock edge.
4. Use `disable iff` with the correct reset polarity as shown in the RTL.
5. Use descriptive labels for every assertion (e.g., `check_grant_mutex`, not `a1`).
6. Add a brief comment above each assertion explaining what it checks.
7. Only assert behaviors that the RTL actually implements. Do not invent \
   signals, states, or protocols that are not present in the code.
8. Focus on QUALITY and CORRECTNESS — 10 correct, meaningful assertions are \
   worth more than 30 trivial or speculative ones.
9. Do NOT wrap output in markdown code fences or add explanation outside the code.
10. Keep comments minimal — one short line per assertion. Do NOT include large \
   comment blocks, file headers, or explanations of your approach.
REFERENCE EXAMPLE — this is the style and quality level to target:
```systemverilog
module manual (
    input logic CLK,
    input logic RESETn,
    input logic QREQn,
    input logic QACCEPTn,
    input logic QDENY,
    input logic QACTIVE
);
    ///// Handshake rules /////
    // QREQn can only transition from HIGH to LOW when QACCEPTn is HIGH and QDENY is LOW.
    handshake_1: assume property (
        @(posedge CLK) disable iff (!RESETn) $fell(QREQn) |-> (QACCEPTn == 1'b1) && (QDENY == 1'b0)
    );
    // QACCEPTn can only transition from HIGH to LOW when QREQn is LOW and QDENY is LOW.
    handshake_3: assert property (
        @(posedge CLK) disable iff (!RESETn) $fell(QACCEPTn) |-> (QREQn == 1'b0) && (QDENY == 1'b0)
    );
    // QDENY can only transition from LOW to HIGH when QREQn is LOW and QACCEPTn is HIGH.
    handshake_6: assert property (
        @(posedge CLK) disable iff (!RESETn) $rose(QDENY) |-> (QREQn == 1'b0) && (QACCEPTn == 1'b1)
    );
    ///// Device reset /////
    // At reset assertion, a device must drive both QACCEPTn and QDENY LOW.
    reset: assert property (
        @(posedge CLK) !RESETn |-> (QACCEPTn == 1'b0) && (QDENY == 1'b0)
    );
endmodule
```
Note the pattern: module wrapper with DUT ports as inputs, descriptive labels, \
comments explaining intent, proper clocking and reset disable on every property, \
and appropriate use of assume vs assert."""


def make_user_prompt(rtl_code: str) -> str:
    return f"""Analyze the following RTL module carefully. Identify:
- The clock(s) and reset signal(s), including reset polarity
- Whether the logic is sequential, combinational, or mixed
- The key signals, interfaces, and functional behaviors
Then generate a complete .sv assertion module following the style shown in \
your reference example. Only write assertions for behaviors that are actually \
present in this RTL — do not guess or assume functionality that isn't there. \
For combinational logic, still use clocked assertions (@(posedge clk)).
RTL module:
```verilog
{rtl_code}
```"""

In [ ]:
def format_chat(example):
    """Convert a dataset row into Qwen chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(example["rtl"])},
        {"role": "assistant", "content": example["sva"]},
    ]
    return {"messages": messages}

train_dataset = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
val_dataset = dataset["validation"].map(format_chat, remove_columns=dataset["validation"].column_names)

print(f"Formatted — Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nSample messages[0] role: {train_dataset[0]['messages'][0]['role']}")
print(f"Sample messages[1] role: {train_dataset[0]['messages'][1]['role']}")
print(f"Sample messages[2] role: {train_dataset[0]['messages'][2]['role']}")

## 4. Load Model (4-bit Quantized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = False  # required for gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"

# Qwen uses <|endoftext|> as pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

## 5. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
lengths = [
    len(tokenizer.encode(
        tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    ))
    for ex in train_dataset.select(range(200))
]
print(f"p50={sorted(lengths)[100]} p95={sorted(lengths)[190]} max={max(lengths)}")

In [ ]:
print(train_dataset[0])
print(train_dataset[0]["messages"])

## 6. Train

In [ ]:
import trl
print(trl.__version__)

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,

    # Epochs & batching
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    # Optimizer
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    optim="paged_adamw_8bit",

    # Sequence
    max_length=MAX_SEQ_LENGTH,

    # Precision
    bf16=True,
    fp16=False,

    # Evaluation & saving
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=10,
    report_to="wandb",

    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Train on completions only — masks prompt tokens, only backprops through assistant response
    completion_only_loss=True,

    # Dataset
    dataset_kwargs={"skip_prepare_dataset": False},
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print(f"Training {RUN_NAME}...")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Total steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

In [ ]:
trainer.train()

## 7. Save Adapter

In [ ]:
# Save the best adapter
final_path = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Adapter saved to: {final_path}")

## 8. Quick Sanity Check

Generate SVA for one test sample to make sure the adapter is working.

In [ ]:
from peft import PeftModel

# Reload for inference (if needed after training)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
# model = PeftModel.from_pretrained(model, final_path)

model.eval()

# Grab one test example
import random
test_example_raw = random.choice(test_raw)  # test_raw is already in memory from Section 3
test_example = {"rtl": test_example_raw["Code"], "sva": test_example_raw["Assertion"]}

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": make_user_prompt(test_example["rtl"])},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.1,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 80)
print("GENERATED SVA:")
print("=" * 80)
print(generated)

In [ ]:
wandb.finish()